# Algoritmos Genéticos

## Algoritmos evolucionários x algoritmos genéticos
- **Algoritmos evolucionários (EA)**: modelos computacionais dos processos naturais de evolução. Simulam a evolução das espécies, baseados na "sobrevivência do mais apto", auto organização e comportamento adaptativo.
- **Algoritmos genéticos (GA)**: um ramo dos algoritmos evolucionários. A ideia é encontrar soluções cada vez melhores a partir da evolução das gerações anteriores.

## Ciclo de um algoritmo genético
1. Gerar população inicial
2. Avaliar população (função de avaliação / fitness)
3. Verificar critério de parada
   - Se atingido: listar os melhores indivíduos e finalizar
   - Caso contrário: continuar
4. Selecionar pais (indivíduos mais aptos têm mais chance de ser escolhidos)
5. Crossover (reprodução) entre os pais selecionados
6. Mutação dos filhos gerados
7. Avaliar a nova população
8. Definir a população sobrevivente (descarta a população antiga)
9. Voltar ao passo 3

Cada passagem completa por esse ciclo é chamada de **geração**.

## Problema utilizado como exemplo
O exemplo deste notebook é baseado no problema de organizar a carga de um caminhão de mudança: a partir de uma lista de produtos (cada um com um espaço em m³ e um valor em R$), o objetivo é escolher quais produtos levar de forma a **maximizar o valor total da carga sem ultrapassar a capacidade máxima do caminhão (3 m³)**.

In [56]:
import pandas as pd
import numpy as np
from numpy.random import random

In [57]:
df_produtos = pd.read_csv("produtos.csv")
df_produtos

,nome,espaco,valor
0,Geladeira Dako,0.75100,999.90
1,Iphone 6,0.00009,2199.12
2,TV 55',0.40000,4346.99
3,TV 50',0.29000,3999.90
4,TV 42',0.20000,2999.00
5,Notebook Dell,0.00350,2499.90
6,Ventilador Panasonic,0.49600,199.90
7,Microondas Electrolux,0.04240,308.66
8,Microondas LG,0.05440,429.90
9,Microondas Panasonic,0.03190,299.29


## Indivíduos
- Cada indivíduo representa uma solução específica para o problema
- Um conjunto de indivíduos forma uma população
- O **cromossomo** representa a solução em si: neste problema, é um vetor binário (0 ou 1) com um valor para cada produto, indicando se ele está (1) ou não (0) na carga do caminhão
- O indivíduo pode ser o próprio cromossomo (forma mais simples) ou pode conter o cromossomo como atributo — é essa segunda abordagem que a classe `Individuo` abaixo utiliza

In [58]:

class Individuo():
    
    def __init__(self, produtos: list[str], espacos: list[float], valores: list[float], limite_espacos: int, geracao: int = 0):
        """
            INDIVÍDUO
                - Representa uma solução candidata para o problema
                - O cromossomo é gerado aleatoriamente como um vetor booleano (0 ou 1),
                  com um gene para cada produto, indicando se ele entra (True) ou não (False) na carga
                - 'geracao' indica em qual geração do algoritmo genético esse indivíduo foi criado
        """
        
        self.produtos = np.array(produtos)
        self.espacos = np.array(espacos)
        self.valores = np.array(valores)
        self.limite_espacos = limite_espacos
        self.nota_avaliacao = 0
        self.espaco_usado = 0
        self.geracao = geracao
        self.cromossomo = np.array([
            random() >= 0.5
            for _ in range(len(self.espacos))
        ])
    
    
    def avaliacao(self):
        """
            FUNÇÃO DE AVALIAÇÃO (FITNESS)
                - Medida de qualidade para saber como o cromossomo resolve o problema
                - Se é uma solução aceitável e se pode ser utilizada para a evolução
                - Soma o valor de todos os produtos selecionados (genes = True) e
                  soma também o espaço ocupado por eles
                - Caso a soma dos espaços ultrapasse o limite (capacidade do caminhão),
                  a nota é rebaixada para 1, penalizando soluções inviáveis
        """
        
        nota = 0
        soma_espacos = 0
        
        for i in range(len(self.cromossomo)):
            if self.cromossomo[i]:
                nota += self.valores[i]
                soma_espacos += self.espacos[i]
        
        if soma_espacos > self.limite_espacos:
            nota = 1                            # Rebaixa a nota caso a soma dos espaços ultrapassar o limite
        
        self.nota_avaliacao = nota
        self.espaco_usado = soma_espacos


    def crossover(self, outro_individuo):
        """
            CROSSOVER (REPRODUÇÃO) DE UM PONTO
                - Combina pedaços do cromossomo de dois pais (self e outro_individuo)
                  para gerar dois filhos mais aptos, fazendo a população evoluir ao longo das gerações
                - Representa a reprodução sexuada: diferente da reprodução assexuada
                  (onde o filho seria idêntico ao pai), aqui há criação de diversidade
                - Um ponto de corte é escolhido aleatoriamente no cromossomo:
                    - Filho 1 = início do cromossomo de 'outro_individuo' + final do cromossomo de 'self'
                    - Filho 2 = início do cromossomo de 'self' + final do cromossomo de 'outro_individuo'
                - Os filhos são criados na geração seguinte (geracao + 1)
        """
        
        corte = round(random() * len(self.cromossomo))
        
        filho1 = np.concatenate((outro_individuo.cromossomo[:corte], self.cromossomo[corte:]))
        filho2 = np.concatenate((self.cromossomo[:corte], outro_individuo.cromossomo[corte:]))
        
        filhos = (Individuo(self.produtos, self.espacos, self.valores, self.limite_espacos, self.geracao + 1), Individuo(self.produtos, self.espacos, self.valores, self.limite_espacos, self.geracao + 1))
        
        filhos[0].cromossomo = filho1
        filhos[1].cromossomo = filho2
        
        return filhos
    
    
    def mutacao(self, taxa_mutacao: float):
        """
            MUTAÇÃO
                - Cria diversidade na população, alterando aleatoriamente genes (bits) do cromossomo
                - É aplicada com menor frequência que o crossover, assim como ocorre na natureza
                - 'taxa_mutacao' é a probabilidade (extremamente baixa) de cada gene ser invertido
                  (True -> False ou False -> True)
        """
        
        for i in range(len(self.cromossomo)):
            if random() < taxa_mutacao:
                self.cromossomo[i] = not self.cromossomo[i]
        
        return self


    def __str__(self) -> str:
        """
            Retorna uma representação legível do indivíduo, listando apenas os
            produtos cujo gene no cromossomo é True (ou seja, que fazem parte da carga)
        """
        
        return "\n".join([
            f"{self.produtos[i]}: R${self.valores[i]}"
            for i in range(len(self.produtos))
            if self.cromossomo[i]
        ])

## População
- Uma **população** é um conjunto de indivíduos (cromossomos), cada um representando uma combinação diferente de produtos para a carga do caminhão
- Abaixo, criamos alguns indivíduos isoladamente (`individuo1`, `individuo2`) para demonstrar como o cromossomo, a avaliação e o crossover funcionam — esses indivíduos seriam, na prática, membros de uma população maior, que é avaliada e evolui a cada geração

### Testando a implementação da classe `Individuo`

In [59]:
produtos = df_produtos["nome"].to_list()
espacos = df_produtos["espaco"].to_list()
valores = df_produtos["valor"].to_list()
limite_espacos = 3

individuo1 = Individuo(produtos, espacos, valores, limite_espacos)
print(individuo1)
individuo2 = Individuo(produtos, espacos, valores, limite_espacos)
print(f"\n{individuo2}")

Geladeira Dako: R$999.9
Iphone 6: R$2199.12
TV 55': R$4346.99
TV 50': R$3999.9
TV 42': R$2999.0
Notebook Dell: R$2499.9
Ventilador Panasonic: R$199.9
Microondas Electrolux: R$308.66
Microondas Panasonic: R$299.29
Geladeira Brastemp: R$849.0
Notebook Lenovo: R$1999.9

Iphone 6: R$2199.12
TV 55': R$4346.99
TV 50': R$3999.9
TV 42': R$2999.0
Notebook Dell: R$2499.9
Microondas Electrolux: R$308.66
Geladeira Brastemp: R$849.0
Notebook Asus: R$3999.0


### Cromossomo

In [60]:
individuo1.cromossomo

array([ True,  True,  True,  True,  True,  True,  True,  True, False,
        True,  True, False,  True, False])

In [61]:
individuo2.cromossomo

array([False,  True,  True,  True,  True,  True, False,  True, False,
       False,  True, False, False,  True])

### Função de Avaliação

In [62]:
individuo1.avaliacao()

print(f"Nota: {individuo1.nota_avaliacao}")
print(f"Espaço usado: {individuo1.espaco_usado}")

Nota: 1
Espaço usado: 3.3478899


### Crossover

In [63]:
filhos = individuo1.crossover(individuo2)
print("PAIS:")
print(individuo1.cromossomo)
print(individuo2.cromossomo)

print("\nFILHOS:")
print(filhos[0].cromossomo)
print(filhos[1].cromossomo)


PAIS:
[ True  True  True  True  True  True  True  True False  True  True False
  True False]
[False  True  True  True  True  True False  True False False  True False
 False  True]

FILHOS:
[False  True  True  True  True  True False  True False  True  True False
  True False]
[ True  True  True  True  True  True  True  True False False  True False
 False  True]


### Mutação

In [64]:
print(f"Indivíduo 1 antes:\n{individuo1.cromossomo}")
individuo1.mutacao(0.05)
print(f"Indivíduo 1 depois:\n{individuo1.cromossomo}")

print(f"\nIndivíduo 2 antes:\n{individuo2.cromossomo}")
individuo2.mutacao(0.05)
print(f"Indivíduo 2 depois:\n{individuo2.cromossomo}")

Indivíduo 1 antes:
[ True  True  True  True  True  True  True  True False  True  True False
  True False]
Indivíduo 1 depois:
[ True False  True  True  True  True  True  True False  True  True False
 False False]

Indivíduo 2 antes:
[False  True  True  True  True  True False  True False False  True False
 False  True]
Indivíduo 2 depois:
[False  True  True  True False False False  True False False  True False
 False  True]


In [ ]:
class AlgoritmoGenetico():
    
    def __init__(self, tamanho_populacao: int):
        
        self.tamanho_populacao = tamanho_populacao
        self.populacao = []
        self.geracao = 0
        self.melhor_solucao = 0
    
    
    def inicializar_populacao(self, produtos, espacos, valores, limite_espacos):
        
        for _ in range(self.tamanho_populacao):
            self.populacao.append(Individuo(produtos, espacos, valores, limite_espacos))
        
        self.populacao = np.array(self.populacao)
        self.melhor_solucao = self.populacao[0]
    
    
    def ordenar_populacao(self):
        
        self.populacao = sorted(
            self.populacao,
            key=lambda populacao: populacao.nota_avaliacao,
            reverse=True
        )
    
    
    def __str__(self):
        
        return "\n".join([
            f"--- INDIVÍDUO {n + 1} ---\n"
            f"Produtos:\n{individuo.produtos}\n"
            f"Espaços:\n{individuo.espacos}\n"
            f"Valores:\n{individuo.valores}\n"
            f"Cromossomo:\n{individuo.cromossomo}\n"
            for n, individuo in enumerate(self.populacao)
        ])

In [66]:
tamanho_populacao = 5
ag = AlgoritmoGenetico(tamanho_populacao)
ag.inicializar_populacao(produtos, espacos, valores, limite_espacos)
print(ag)

--- INDIVÍDUO 1 ---
Produtos:
['Geladeira Dako' 'Iphone 6' "TV 55'" "TV 50'" "TV 42'" 'Notebook Dell'
 'Ventilador Panasonic' 'Microondas Electrolux' 'Microondas LG'
 'Microondas Panasonic' 'Geladeira Brastemp' 'Geladeira Consul'
 'Notebook Lenovo' 'Notebook Asus']
Espaços:
[7.51e-01 8.99e-05 4.00e-01 2.90e-01 2.00e-01 3.50e-03 4.96e-01 4.24e-02
 5.44e-02 3.19e-02 6.35e-01 8.70e-01 4.98e-01 5.27e-01]
Valores:
[ 999.9  2199.12 4346.99 3999.9  2999.   2499.9   199.9   308.66  429.9
  299.29  849.   1199.89 1999.9  3999.  ]
Cromossomo:
[ True False  True False  True  True False False  True  True  True  True
 False  True]

--- INDIVÍDUO 2 ---
Produtos:
['Geladeira Dako' 'Iphone 6' "TV 55'" "TV 50'" "TV 42'" 'Notebook Dell'
 'Ventilador Panasonic' 'Microondas Electrolux' 'Microondas LG'
 'Microondas Panasonic' 'Geladeira Brastemp' 'Geladeira Consul'
 'Notebook Lenovo' 'Notebook Asus']
Espaços:
[7.51e-01 8.99e-05 4.00e-01 2.90e-01 2.00e-01 3.50e-03 4.96e-01 4.24e-02
 5.44e-02 3.19e-02 6.35e-

## Seleção dos indivíduos
- Os operadores genéticos (crossover e mutação) são aplicados sobre indivíduos **selecionados** dentro da população
- Indivíduos mais aptos (nota de avaliação maior) devem ser selecionados com mais frequência, para que suas características predominem na nova população
- Porém, indivíduos menos aptos não devem ser completamente descartados — assim como na seleção natural, pais menos capazes também geram descendentes
- Se apenas os melhores indivíduos forem mantidos, a população tende a ficar cada vez mais homogênea, perdendo diversidade

### Método da roleta viciada
- Cada cromossomo recebe uma fatia da "roleta" proporcional à sua nota de avaliação (quanto maior a nota, maior a fatia)
- A roleta é "girada" para escolher os pais que participarão do crossover, dando mais chances aos mais aptos sem eliminar os demais
- **Elitismo**: técnica que preserva sempre os melhores indivíduos de uma geração para a próxima, garantindo que o melhor resultado encontrado nunca seja perdido

> Próximo passo do notebook: implementar a função de seleção (roleta viciada) e o laço principal do algoritmo genético, unindo os conceitos de população, avaliação, seleção, crossover e mutação em um ciclo completo de gerações.